In [2]:
import numpy as np 
import pandas as pd
import matplotlib.pyplot as plt

In [5]:
!pip install nnfs
import nnfs
from nnfs.datasets import vertical_data


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Backpropagation from Scratch  
## Sigmoid + Binary Cross-Entropy + Adam Optimizer (Using `nnfs` Vertical Data)

---

## 1. Problem Setup

We train a **binary classifier** using:
- Dataset: `nnfs.datasets.vertical_data`
- Activation: **Sigmoid**
- Loss: **Binary Cross-Entropy**
- Optimizer: **Adam**
- Implementation: **Pure NumPy**

This demonstrates how backpropagation works **numerically and algorithmically**.


In [6]:
import numpy as np
import nnfs
from nnfs.datasets import vertical_data

nnfs.init()

In [7]:
# -----------------------------
# Dataset
# -----------------------------
X, y = vertical_data(samples=1000, classes=3)

# One-hot encode labels
num_classes = 3
y_onehot = np.eye(num_classes)[y]

N, D = X.shape   # samples, features

In [8]:
def relu(z):
    return np.maximum(0, z)

def relu_derivative(z):
    return (z > 0).astype(float)

def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    exp_z = np.exp(z)
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

In [9]:
def cross_entropy(y_true, y_pred):
    eps = 1e-7
    y_pred = np.clip(y_pred, eps, 1 - eps)
    return -np.mean(np.sum(y_true * np.log(y_pred), axis=1))

In [10]:
np.random.seed(0)

H1 = 64
H2 = 32

W1 = 0.01 * np.random.randn(D, H1)
b1 = np.zeros((1, H1))

W2 = 0.01 * np.random.randn(H1, H2)
b2 = np.zeros((1, H2))

W3 = 0.01 * np.random.randn(H2, num_classes)
b3 = np.zeros((1, num_classes))

In [11]:
def init_adam(param):
    return np.zeros_like(param), np.zeros_like(param)

mW1, vW1 = init_adam(W1)
mW2, vW2 = init_adam(W2)
mW3, vW3 = init_adam(W3)

mb1, vb1 = init_adam(b1)
mb2, vb2 = init_adam(b2)
mb3, vb3 = init_adam(b3)

lr = 0.001
beta1 = 0.9
beta2 = 0.999
eps = 1e-8
t = 0

In [12]:
epochs = 1000

for epoch in range(epochs):
    # -----------------------------
    # Forward pass
    # -----------------------------
    z1 = X @ W1 + b1
    a1 = relu(z1)

    z2 = a1 @ W2 + b2
    a2 = relu(z2)

    z3 = a2 @ W3 + b3
    y_hat = softmax(z3)

    loss = cross_entropy(y_onehot, y_hat)

    # -----------------------------
    # Backpropagation
    # -----------------------------

    # 🔥 Softmax + Cross-Entropy derivative
    dZ3 = y_hat - y_onehot

    dW3 = a2.T @ dZ3 / N
    db3 = np.sum(dZ3, axis=0, keepdims=True) / N

    dA2 = dZ3 @ W3.T
    dZ2 = dA2 * relu_derivative(z2)

    dW2 = a1.T @ dZ2 / N
    db2 = np.sum(dZ2, axis=0, keepdims=True) / N

    dA1 = dZ2 @ W2.T
    dZ1 = dA1 * relu_derivative(z1)

    dW1 = X.T @ dZ1 / N
    db1 = np.sum(dZ1, axis=0, keepdims=True) / N

    # -----------------------------
    # Adam update
    # -----------------------------
    t += 1

    def adam_update(W, dW, mW, vW):
        mW = beta1 * mW + (1 - beta1) * dW
        vW = beta2 * vW + (1 - beta2) * (dW ** 2)

        m_hat = mW / (1 - beta1 ** t)
        v_hat = vW / (1 - beta2 ** t)

        W -= lr * m_hat / (np.sqrt(v_hat) + eps)
        return W, mW, vW

    W3, mW3, vW3 = adam_update(W3, dW3, mW3, vW3)
    W2, mW2, vW2 = adam_update(W2, dW2, mW2, vW2)
    W1, mW1, vW1 = adam_update(W1, dW1, mW1, vW1)

    b3, mb3, vb3 = adam_update(b3, db3, mb3, vb3)
    b2, mb2, vb2 = adam_update(b2, db2, mb2, vb2)
    b1, mb1, vb1 = adam_update(b1, db1, mb1, vb1)

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")

Epoch 0, Loss: 1.0986
Epoch 100, Loss: 0.7670
Epoch 200, Loss: 0.2619
Epoch 300, Loss: 0.1670
Epoch 400, Loss: 0.1493
Epoch 500, Loss: 0.1447
Epoch 600, Loss: 0.1431
Epoch 700, Loss: 0.1424
Epoch 800, Loss: 0.1420
Epoch 900, Loss: 0.1416


In [13]:
print("\nFinal Loss:", loss)
print("Mean W1:", np.mean(W1))
print("Mean b1:", np.mean(b1))
print("Mean W3:", np.mean(W3))


Final Loss: 0.14137192
Mean W1: 0.11461015
Mean b1: 0.14665322
Mean W3: 0.017649729


In [16]:
print("new weight w1 : ", W1[0, 0])
print("new bias b1 : ", b1[0, 0])
print("new weight w3 : ", W3[0, 0])
print("new bias b3 : ", b3[0, 0])

new weight w1 :  0.41536686
new bias b1 :  -0.15049961
new weight w3 :  0.010941669
new bias b3 :  -0.26439923
